<a href="https://colab.research.google.com/github/arnavnigam31/DeepLearningLab/blob/main/experiment9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

df = pd.read_csv("/content/Amazon_Reviews.csv", engine="python")

df = df[["Review Text", "Rating"]]

df["Rating"] = df["Rating"].astype(str).str.extract(r'(\d+)')

df = df.dropna(subset=["Rating"])

df["Rating"] = df["Rating"].astype(int)

df = df[df["Rating"] != 3]

df["label"] = df["Rating"].apply(lambda x: 1 if x >= 4 else 0)

texts = df["Review Text"].astype(str).tolist()
labels = df["label"].tolist()

texts = df["Review Text"].astype(str).tolist()
labels = df["label"].tolist()

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
max_len = 128

class ReviewDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "label": torch.tensor(self.labels[idx])
        }

dataset = ReviewDataset(texts, labels)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = self.embedding(x)
        _, h = self.gru(x)
        h = h.squeeze(0)
        out = self.fc(h)
        return out

model = GRUClassifier(tokenizer.vocab_size, 128, 256).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(3):
    total_loss = 0

    for batch in loader:
        x = batch["input_ids"].to(device)
        y = batch["label"].to(device)

        optimizer.zero_grad()

        preds = model(x)

        loss = criterion(preds, y)
        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch+1, "Loss:", total_loss/len(loader))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Epoch: 1 Loss: 0.32285240247089964
Epoch: 2 Loss: 0.16679613871378868
Epoch: 3 Loss: 0.11122774882183045


In [9]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in loader:
        x = batch["input_ids"].to(device)
        y = batch["label"].to(device)

        outputs = model(x)
        preds = torch.argmax(outputs, dim=1)

        correct += (preds == y).sum().item()
        total += y.size(0)

accuracy = correct / total
print("Accuracy:", accuracy)

Accuracy: 0.9770946950917204
